# Comparing Local LLMs with Ollama

In this lab, you will run small local language models through Ollama and compare their behaviour on short synthetic social and health notes.

The emphasis is model intuition: model size, prompt style, temperature, valid labels, and what makes a local output usable or not.


## How to Use This Notebook

This notebook is designed to run standalone on a Linux teaching laptop, including an 11th gen Framework laptop. Setup 3 installs Ollama if needed, starts the local Ollama server if it is not already running, and pulls two small CPU-friendly models.

The first run can take several minutes while Ollama and model weights download. Use synthetic examples only; do not paste confidential service-user, patient, client, or institutional data into the notebook.


## Setup: Small Local Model Lab

The setup is intentionally short. It defines two small local models, a small labelled dataset, and one repeated-call function for Ollama's local chat endpoint.


## Model Intuition: What Are We Comparing?

Ollama is the local runtime. It downloads a packaged model, keeps it on the laptop, and serves a small HTTP API at `localhost:11434`. Ollama did not train these models; it is the tool we use to run them locally.

The two default models are both small enough for a CPU-only teaching laptop, but they come from different model families and make different trade-offs.

- `smollm2:135m` comes from Hugging Face's Smol Models Research team. The SmolLM2 family has 135M, 360M, and 1.7B parameter versions designed for on-device use. The 135M base model was trained on about 2 trillion tokens from sources including FineWeb-Edu, DCLM, The Stack, and filtered datasets. The instruct version was then tuned with supervised examples and preference training. In practice: fast and small, but more likely to ignore exact format instructions or miss nuance.
- `qwen2.5:0.5b` comes from Qwen, the Alibaba Cloud model family. Qwen2.5 includes base and instruction-tuned models from 0.5B to 72B parameters. This notebook uses the small instruction-tuned 0.5B version, a causal language model trained with pretraining and post-training. In practice: still laptop-friendly, usually better at following instructions than the 135M model, but slower and more memory-hungry.
- The Ollama tags are local packaged versions of these model families. Speed, memory use, and exact behaviour depend on the tag, quantization, laptop, and Ollama version.

The key lesson is not that one model is always best. A local model should be judged against the task: speed, memory use, valid output format, and whether its mistakes are acceptable for the workflow.

Sources: [SmolLM2 model card](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct), [Qwen2.5 model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), [Ollama API docs](https://github.com/ollama/ollama/blob/main/docs/api.md).

### Setup 1: Imports, Models, and Labels

`MODELS` controls which local models are compared. The defaults are intentionally small so they can run on a CPU-only laptop.


In [ ]:
# Setup 1
import json
import subprocess
import time

import requests

OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"
OLLAMA_TAGS_URL = "http://localhost:11434/api/tags"
MODELS = ["smollm2:135m", "qwen2.5:0.5b"]
DEFAULT_MODEL = MODELS[0]
ALLOWED_LABELS = ["access_barrier", "housing_stress", "health_need", "other"]


### Setup 2: Synthetic Notes and Prompts

The labels are deliberately simple. The point is to compare local model behaviour, not to build a production classifier.


In [ ]:
# Setup 2
evaluation_notes = [
    {"id": "L001", "text": "Bus route cancelled; resident missed clinic appointment.", "human_label": "access_barrier"},
    {"id": "L002", "text": "Rent arrears after reduced work hours; eviction notice received.", "human_label": "housing_stress"},
    {"id": "L003", "text": "Medication side effects and dizziness; needs GP follow-up.", "human_label": "health_need"},
    {"id": "L004", "text": "Resident praised the new drop-in coffee morning.", "human_label": "other"},
    {"id": "L005", "text": "Care worker could not enter because the tower block intercom failed.", "human_label": "access_barrier"},
    {"id": "L006", "text": "Temporary accommodation has damp; family asks for housing advice.", "human_label": "housing_stress"},
    {"id": "L007", "text": "Neighbour reports the resident is isolated and missing meals.", "human_label": "health_need"},
    {"id": "L008", "text": "Community centre extended its opening hours for winter activities.", "human_label": "other"},
]

def label_prompt(text, strict=True):
    if strict:
        return "Return exactly one label: access_barrier, housing_stress, health_need, or other.\nText: " + text
    return "Classify the text, give the label, and briefly explain your reasoning.\nText: " + text

def normalise_label(text):
    cleaned = text.strip().lower().replace("-", "_").replace(" ", "_")
    for label in ALLOWED_LABELS:
        if label in cleaned:
            return label
    return "invalid"


### Optional: macOS and Windows Install Cells

The next two cells are intentionally commented out. Use one only if you are running the notebook on that operating system and the Linux setup cell below is not appropriate.

In [ ]:
# macOS only: leave commented unless you are running this notebook on a Mac.
# Official Ollama install command:
# curl -fsSL https://ollama.com/install.sh | sh
#
# To run from this notebook, uncomment the three lines below.
# import subprocess
# subprocess.run(["bash", "-lc", "command -v ollama >/dev/null || curl -fsSL https://ollama.com/install.sh | sh"], check=True)
# subprocess.run(["ollama", "--version"], check=True)

In [ ]:
# Windows only: leave commented unless you are running this notebook on Windows.
# Official Ollama PowerShell install command:
# irm https://ollama.com/install.ps1 | iex
#
# To run from this notebook, uncomment the lines below.
# import subprocess
# subprocess.run([
#     "powershell",
#     "-NoProfile",
#     "-ExecutionPolicy",
#     "Bypass",
#     "-Command",
#     "if (-not (Get-Command ollama -ErrorAction SilentlyContinue)) { irm https://ollama.com/install.ps1 | iex }",
# ], check=True)
# subprocess.run(["ollama", "--version"], check=True)

### Setup 3: Linux Install, Start Server, Pull Models

In [ ]:
# Setup 3
print("Installing Ollama on Linux if it is missing...")
subprocess.run(
    ["bash", "-lc", "command -v ollama >/dev/null || curl -fsSL https://ollama.com/install.sh | sh"],
    check=True,
)

print("Checking Ollama command...")
subprocess.run(["ollama", "--version"], check=True)

print("Starting Ollama server if needed...")
try:
    requests.get(OLLAMA_TAGS_URL, timeout=2).raise_for_status()
    print("Ollama server is already responding.")
except requests.exceptions.RequestException:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)
    requests.get(OLLAMA_TAGS_URL, timeout=10).raise_for_status()
    print("Ollama server started.")

for model in MODELS:
    print("Pulling or checking model:", model)
    subprocess.run(["ollama", "pull", model], check=True)

print("Ready to call local models:", MODELS)


### Setup 4: One Small Repeated-Call Function

Question 2 writes out the first request directly. Later cells use `ask_ollama(...)` so we can focus on model comparisons rather than repeating the same HTTP request.


In [ ]:
# Setup 4
def ask_ollama(model, prompt, temperature=0.0, num_predict=40):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }
    start = time.time()
    response = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=120)
    if response.status_code != 200:
        raise RuntimeError(response.text)
    data = response.json()
    return {
        "model": model,
        "temperature": temperature,
        "elapsed_seconds": time.time() - start,
        "payload": payload,
        "json": data,
        "text": data["message"]["content"].strip(),
    }


## Part A: First Local Call (Approx. 25 minutes)

Before comparing models, inspect one request and one response. A local model call still has a payload, parameters, and generated text.


### Question 1: Inspect the Task

Look at the synthetic notes and labels before running a model. A model comparison is only meaningful when the task is clear.


In [ ]:
# Answer 1
for note in evaluation_notes:
    print(note["id"], "|", note["human_label"], "|", note["text"])

print("Allowed labels:", ALLOWED_LABELS)


### Question 2: Make One Local API Call

This first call is written out. Notice the same ingredients as a hosted API call: URL, payload, model name, prompt, settings, JSON response, and parsed text.


In [ ]:
# Answer 2
selected_note = evaluation_notes[0]
prompt = label_prompt(selected_note["text"])

payload = {
    "model": DEFAULT_MODEL,
    "messages": [{"role": "user", "content": prompt}],
    "stream": False,
    "options": {"temperature": 0.0, "num_predict": 40},
}

print("URL:", OLLAMA_CHAT_URL)
print("Payload:")
print(json.dumps(payload, indent=2))

start = time.time()
response = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=120)
if response.status_code != 200:
    raise RuntimeError(response.text)

first_json = response.json()
first_text = first_json["message"]["content"].strip()
first_result = {
    "model": DEFAULT_MODEL,
    "text": first_text,
    "label": normalise_label(first_text),
    "elapsed_seconds": time.time() - start,
}

print("LOCAL MODEL CALLED: YES")
print(first_result)


### Question 3: Interpret the First Output

A useful output is not just text. It should be valid for the task and comparable with the human label.


In [ ]:
# Answer 3
first_result["human_label"] = selected_note["human_label"]
first_result["match"] = first_result["label"] == selected_note["human_label"]

print(json.dumps(first_result, indent=2))


## Part B: Compare Local Models (Approx. 45 minutes)

Now each model gets the same notes, prompt style, temperature, and output limit. This makes the comparison fair.


### Question 4: Run the Same Notes Through Each Model

Every row is one model-note pair. If a model is slower, less consistent, or more verbose, that is part of the comparison.


In [ ]:
# Answer 4
comparison_rows = []

for model in MODELS:
    print("Running model:", model)
    for note in evaluation_notes:
        result = ask_ollama(
            model=model,
            prompt=label_prompt(note["text"]),
            temperature=0.0,
            num_predict=40,
        )
        model_label = normalise_label(result["text"])
        row = {
            "model": model,
            "note_id": note["id"],
            "human_label": note["human_label"],
            "model_text": result["text"],
            "model_label": model_label,
            "match": model_label == note["human_label"],
            "elapsed_seconds": round(result["elapsed_seconds"], 2),
        }
        comparison_rows.append(row)
        print(row)


### Question 5: Score the Model Outputs

Tiny synthetic scores are not research evidence. They are a structured way to compare behaviour under the same task.


In [ ]:
# Answer 5
model_scores = {}

for model in MODELS:
    total = 0
    correct = 0
    valid = 0
    for row in comparison_rows:
        if row["model"] == model:
            total = total + 1
            if row["match"]:
                correct = correct + 1
            if row["model_label"] in ALLOWED_LABELS:
                valid = valid + 1
    model_scores[model] = {
        "n": total,
        "accuracy": correct / total,
        "valid_label_rate": valid / total,
    }

print(json.dumps(model_scores, indent=2))


### Question 6: Inspect Mistakes

Do not stop at a score. Read the mistakes and ask what kind of instruction-following problem they show.


In [ ]:
# Answer 6
mistakes = []
for row in comparison_rows:
    if not row["match"]:
        mistakes.append(row)
        print(row)

if len(mistakes) == 0:
    print("No mistakes on this tiny synthetic set. Try adding harder notes before trusting the model.")


## Part C: Tune Settings and Prompt Style (Approx. 35 minutes)

Local model behaviour changes when you change parameters or prompt style. The goal is to understand the direction of the change, not to search every possible setting.

In the API payload:

- `model` chooses which local model Ollama should run.
- `messages` is the chat history. Here we usually send one user message.
- `options` contains model-generation settings such as `temperature` and `num_predict`.
- `stream=False` tells Ollama to return one complete JSON response instead of many small token-by-token chunks. That is easier to inspect in an introductory notebook.
- `timeout=120` is only a Python `requests` setting. It says how long Python should wait before giving up; it does not change how the model thinks.

The main settings in this lab:

- `temperature` controls randomness. Lower values make the model more predictable and are often better for labels. Higher values can make answers more varied, but also less consistent.
- `num_predict` is the maximum number of output tokens the model is allowed to generate. Tokens are chunks of text, not exactly words. A small value is useful for short labels; a larger value is needed for explanations or summaries.

Other common Ollama settings you may see but do not need to tune here:

- `top_p` and `top_k` control which next-token choices are allowed during sampling. Lower values usually make text more conservative; higher values allow more variety.
- `repeat_penalty` discourages repeated words or phrases.
- `seed` can make output more repeatable for the same prompt and settings.
- `num_ctx` controls how much context the model can consider. This is different from `num_predict`, which controls how much new text it can write.

### Question 7: Compare Two Temperatures

Temperature changes how deterministic the model is. Low temperature is usually better for label-only classification.


In [ ]:
# Answer 7
tuning_model = MODELS[0]
tuning_note = evaluation_notes[1]
temperature_rows = []

for temperature in [0.0, 0.8]:
    result = ask_ollama(
        model=tuning_model,
        prompt=label_prompt(tuning_note["text"]),
        temperature=temperature,
        num_predict=40,
    )
    row = {
        "model": tuning_model,
        "temperature": temperature,
        "text": result["text"],
        "label": normalise_label(result["text"]),
    }
    temperature_rows.append(row)
    print(row)


### Question 8: Compare Strict and Explanation Prompts

Explanation prompts can be useful for human review, but they are harder to score automatically. This is a core trade-off in applied LLM work.


In [ ]:
# Answer 8
prompt_style_rows = []
style_note = evaluation_notes[4]

for strict in [True, False]:
    result = ask_ollama(
        model=tuning_model,
        prompt=label_prompt(style_note["text"], strict=strict),
        temperature=0.0,
        num_predict=80,
    )
    row = {
        "strict_prompt": strict,
        "text": result["text"],
        "normalised_label": normalise_label(result["text"]),
    }
    prompt_style_rows.append(row)
    print(row)


### Question 9: Vary the Output Limit

`num_predict` is Ollama's maximum output-token budget for the model response. Tokens are chunks of text, so 20 tokens is not exactly 20 words. For a label-only task, a low value can be helpful because the answer should be short. For explanations, redactions, or summaries, too low a value can cut the answer off before it finishes.

This is different from context length. `num_predict` limits the output; `num_ctx` controls how much input and conversation history the model can look at.

In [ ]:
# Answer 9
output_limit_rows = []
limit_note = evaluation_notes[0]

for num_predict in [8, 80]:
    result = ask_ollama(
        model=tuning_model,
        prompt=label_prompt(limit_note["text"], strict=False),
        temperature=0.0,
        num_predict=num_predict,
    )
    row = {
        "num_predict": num_predict,
        "text": result["text"],
        "normalised_label": normalise_label(result["text"]),
    }
    output_limit_rows.append(row)
    print(row)


## Part D: Recommendation (Approx. 20 minutes)

A recommendation should connect evidence to the task. Fast is useful, but only if the model follows the required output format.


### Question 10: Recommend a Model and Setting

Choose the model with the best accuracy, then valid-label rate. If scores tie, prefer the smaller or faster model.


In [ ]:
# Answer 10
best_model = None
best_score = -1

for model, scores in model_scores.items():
    score = scores["accuracy"] + scores["valid_label_rate"]
    if score > best_score:
        best_model = model
        best_score = score

recommendation = {
    "recommended_model": best_model,
    "temperature": 0.0,
    "prompt_style": "strict label-only prompt",
    "num_predict": 40,
    "reason": "Prioritise valid labels, task accuracy, and predictable output for classification.",
}
print(json.dumps(recommendation, indent=2))


### Question 11: Final Comparison Report

Write a compact report that names the task, candidate models, settings, evidence, and one limitation.


In [ ]:
# Answer 11
comparison_report = {
    "task": "classify short synthetic social and health notes",
    "candidate_models": MODELS,
    "model_scores": model_scores,
    "recommended_model": recommendation["recommended_model"],
    "recommended_settings": {
        "temperature": recommendation["temperature"],
        "prompt_style": recommendation["prompt_style"],
        "num_predict": recommendation["num_predict"],
    },
    "parameter_checks": {
        "temperatures": temperature_rows,
        "prompt_styles": prompt_style_rows,
        "output_limits": output_limit_rows,
    },
    "limitation": "The evaluation set is tiny and synthetic, so it only demonstrates a comparison workflow.",
}

print(json.dumps(comparison_report, indent=2))


## End of Lab Checklist

By the end of this lab, you should have:

- installed or checked Ollama from the notebook on Linux;
- started or checked the local Ollama server;
- pulled two small CPU-friendly local models;
- described the main difference between the two models;
- inspected one local `/api/chat` payload;
- made real local model calls;
- compared two local models on the same notes;
- normalised outputs to allowed labels;
- calculated simple accuracy and valid-label rates;
- inspected model mistakes;
- compared low and high temperature;
- compared strict and explanation prompts;
- varied the output limit with `num_predict`;
- written a brief model recommendation.
